# Required Imports

In [1]:
import os
import re
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
import numpy as np
from numpy import dot
from numpy.linalg import norm
from collections import defaultdict
from tqdm import tqdm
from PIL import Image

class HardTripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super(HardTripletLoss, self).__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, inputs, targets):
        n = inputs.size(0)
        dist = torch.pow(inputs, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()
        
        # SINTASSI AGGIORNATA PER EVITARE IL DEPRECATION WARNING
        dist.addmm_(inputs, inputs.t(), beta=1, alpha=-2)
        
        dist = dist.clamp(min=1e-12).sqrt()

        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        dist_ap, dist_an = [], []

        for i in range(n):
            dist_ap.append(dist[i][mask[i]].max().unsqueeze(0))
            dist_an.append(dist[i][mask[i] == 0].min().unsqueeze(0))

        dist_ap = torch.cat(dist_ap)
        dist_an = torch.cat(dist_an)
        y = torch.ones_like(dist_an)
        return self.ranking_loss(dist_an, dist_ap, y)

## Parameters

In [2]:
# Set a fixed random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)  # Seed for NumPy random number generator
torch.manual_seed(RANDOM_SEED)  # Seed for PyTorch random number generator

# Set the depth of the network
depth = 32  # Initial depth of the Siamese network
img_channels = 3  # Number of color channels (3 for RGB)
img_w = 172  # Image width
img_h = 128  # Image height

# Determine the computing device (GPU if available, otherwise CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f'device being used: {device}')


# Configuration settings
n_classes = 40 # Number of identities in the training set
n_epochs = 20  # Number of training epochs
train_batch_size = 32  # Batch size for training
val_batch_size = 1  # Batch size for validation
test_batch_size = 1  # Batch size for testing
learning_rate = 0.0005  # Learning rate for the optimizer
backbone = 'siameseNet'  # Backbone network architecture ['siameseNet', 'VGG16', 'VGG19', 'denseNet', 'MobileNetV3', 'efficientNetB0', 'ViT16']

device being used: mps


## Load Probe and Gallery sets

In [3]:
def extract_id_from_imagename(filename):
    match = re.search(r'(\d+[mf])\_', filename) 
    return match.group(1) if match else None

class FlatReIDTrainDataset(Dataset):
    def __init__(self, data_root, transform=None):
        self.data_root = data_root
        self.transform = transform
        self.samples, self.id_to_class = self._load_data()

    def _load_data(self):
        paths = []
        for dirpath, _, filenames in os.walk(self.data_root):
            for f in filenames:
                if f.endswith(('png', 'jpg', 'jpeg')):
                    paths.append(os.path.join(dirpath, f))
        
        unique_ids = sorted(list(set([extract_id_from_imagename(p) for p in paths if extract_id_from_imagename(p)])))
        id_to_class = {orig_id: idx for idx, orig_id in enumerate(unique_ids)}
        
        samples = []
        for p in paths:
            pid = extract_id_from_imagename(p)
            if pid:
                samples.append((p, id_to_class[pid]))
        return samples, id_to_class

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, target = self.samples[idx]
        img = Image.open(img_path)
        if self.transform:
            img = self.transform(img)
        return img, target

class PKSampler(torch.utils.data.Sampler):
    def __init__(self, dataset, P=8, K=4):
        self.dataset = dataset
        self.P = P
        self.K = K
        self.batch_size = self.P * self.K
        
        self.id_to_indices = defaultdict(list)
        for idx, (_, target) in enumerate(self.dataset.samples):
            self.id_to_indices[target].append(idx)
        self.unique_ids = list(self.id_to_indices.keys())

    def __iter__(self):
        random.shuffle(self.unique_ids)
        batches = []
        for i in range(0, len(self.unique_ids), self.P):
            p_ids = self.unique_ids[i:i+self.P]
            if len(p_ids) < self.P:
                break
            batch = []
            for pid in p_ids:
                indices = self.id_to_indices[pid]
                if len(indices) >= self.K:
                    batch.extend(random.sample(indices, self.K))
                else:
                    batch.extend(np.random.choice(indices, size=self.K, replace=True))
            batches.append(batch)
        
        random.shuffle(batches)
        for batch in batches:
            yield from batch

    def __len__(self):
        return (len(self.unique_ids) // self.P) * self.batch_size

# create the val/test dataset
class ProbeGalleryValTest(Dataset):
    def __init__(self, data_root, transform=None):
        self.data_root = data_root  
        self.data = self.read_folder()  
        self.samples = self.create_samples()  
        self.transform = transform  

    def __len__(self):
        return len(self.samples)  

    def __getitem__(self, idx):
      probe_pth, gallery_pths, is_match, probe_target, gallery_targets = self.samples[idx]  
      probe = Image.open(probe_pth)  

      gallery = []
      for item in gallery_pths:
          gallery.append(Image.open(item))

      if self.transform:
          probe = self.transform(probe)
          for i in range(len(gallery)):
              gallery[i] = self.transform(gallery[i])

      return [probe, gallery, is_match, probe_pth, gallery_pths, probe_target, gallery_targets]

    def read_folder(self):
        paths = []  
        labels = []  
        for dirpath, dirnames, filenames in os.walk(self.data_root):
            files = [f for f in filenames if f.split('.')[-1] in ['jpg', 'jpeg', 'png']]
            for item in files:
                label = dirpath.split('/')[-1]  
                if label not in labels:
                    labels.append(label)
                paths.append([f'{dirpath}/{item}', label])  
        return paths

    def create_samples(self):
        probes = [x[0] for x in self.data if x[1] == 'Probe']  
        gallery = [x[0] for x in self.data if x[1] == 'Gallery']  

        samples = []
        for probe in probes:
          probe_id = extract_id_from_imagename(probe)  
          if not probe_id: continue

          is_match = []
          gallery_targets = []
          
          for item in gallery:
            item_id = extract_id_from_imagename(item)  
            if not item_id: continue
            
            is_match.append(1 if probe_id == item_id else -1)
            gallery_targets.append(item_id)

          samples.append([probe, gallery, is_match, probe_id, gallery_targets])
        return samples

In [4]:
# Define the transformation to be applied to the images (converting them to PyTorch tensors)
if backbone == 'ViT16':
    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
    ])
else:
    transform = transforms.Compose([
        transforms.Resize(128), 
        transforms.ToTensor(),
    ])


train_dataset = FlatReIDTrainDataset('./output/ImageData/D1/', transform=transform)
pk_sampler = PKSampler(train_dataset, P=8, K=4) 
train_dataloader = DataLoader(train_dataset, batch_size=32, sampler=pk_sampler, drop_last=True)

test_dataset = ProbeGalleryValTest('./output/ImageData/D2t/', transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)



# Neural Network Models

In [5]:
# Define a Siamese network class inheriting from nn.Module
class BSN(nn.Module):
    def __init__(self, backbone, img_channels, depth, img_w, img_h, n_classes):
        super(BSN, self).__init__()

        self.backbone = backbone
        
        if self.backbone == 'siameseNet':
            # Initialize convolutional and pooling layers
            # First convolutional layer with specified input channels and depth
            self.conv1 = nn.Conv2d(img_channels, depth, kernel_size=3, padding=1)
            # First pooling layer, reduces spatial dimensions by half
            self.pool1 = nn.MaxPool2d(2, 2)
            # Second convolutional layer, doubling the depth
            self.conv2 = nn.Conv2d(depth, depth*2, kernel_size=3, padding=1)
            # Second pooling layer, further reducing spatial dimensions
            self.pool2 = nn.MaxPool2d(2, 2)

            # Calculate the output size after the convolutional and pooling layers
            output_size = depth*2 * img_h//4 * img_w//4
            
            

        if self.backbone == 'VGG19':
            self.model = models.vgg19_bn().features # VGG19 with batch normalization

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)
        
        if self.backbone == 'VGG16':
            self.model = models.vgg16_bn().features # VGG16 with batch normalization

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)

        if self.backbone == 'denseNet':
            self.model = models.densenet121().features # DenseNet121 with batch normalization
            
            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)

        if self.backbone == "MobileNetV3":
            self.model = models.mobilenet_v3_small().features # MobileNetV3 with batch normalization

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)
            
        if self.backbone == 'efficientNetB0':
            self.model = models.efficientnet_b0().features # VGG16 with batch normalization

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, img_h, img_w)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)

        if self.backbone == "ViT16":
            self.model = models.vit_b_16() # ViT16
            self.model.heads = nn.Identity() # remove the classification head

            # Create a dummy input tensor of the correct size
            dummy_input = torch.randn(1, img_channels, 224, 224)

            # Run a forward pass through the convolutional layers
            output = self.model(dummy_input)

            # Calculate the output size
            output_size = output.view(output.size(0), -1).size(1)
        
        self.fc1 = nn.Linear(output_size, 256)
        self.fc2 = nn.Linear(256, 256)
        self.classifier = nn.Linear(256,n_classes)
        self.dropout = nn.Dropout()

    # Define the forward pass for one branch of the Siamese network
    def forward_one(self, x):
        if self.backbone == 'siameseNet':
            x = self.pool1(F.relu(self.conv1(x)))
            x = self.pool2(F.relu(self.conv2(x)))
            x = x.view(x.size(0), -1)
            x = self.dropout(F.relu(self.fc1(x)))
            
            # ESTRAZIONE PULITA: Niente ReLU, niente Dropout
            features = self.fc2(x)
            y = self.classifier(features)
        else:
            x = self.model(x)
            x = x.view(x.size(0), -1)
            x = self.dropout(F.relu(self.fc1(x)))
            
            features = self.fc2(x)
            y = self.classifier(features)
        
        # L2 NORMALIZATION: Cruciale per evitare divisioni per zero e per la Triplet Loss
        features = F.normalize(features, p=2, dim=1)
        
        return features, y

    # Define the forward pass for the whole Siamese network
    def forward(self, input1, input2):
        # Process each input through the network
        output1, output_class1 = self.forward_one(input1)
        output2, output_class2= self.forward_one(input2)
        return output1, output_class1, output2, output_class2

# Training and Testing functions

In [6]:
def compute_ap(pos_ranks):
    """Calcolo corretto della Average Precision"""
    if len(pos_ranks) == 0:
        return 0.0
    ap = 0.0
    for i, pos in enumerate(pos_ranks):
        # i + 1: numero di match corretti trovati finora
        # pos + 1: la posizione (rank) in cui è stato trovato
        ap += (i + 1) / (pos + 1)
    return ap / len(pos_ranks)

def testing_step(model, test_dataloader):
    # Switch the model to evaluation mode. This turns off specific layers/features like dropout.
    model.eval()

    # Initialize counters for correct predictions at different ranks
    correct1, correct2, correct3, correct4, correct5 = 0, 0, 0, 0, 0
    correct6, correct7, correct8, correct9, correct10 = 0, 0, 0, 0, 0
    samples = 0  
    
    # Set the rank limit for evaluation
    K = 10
    AP_scores = []  
    rank_matrix = []

    # Disable gradient calculation for efficiency and to prevent changes to the model
    with torch.no_grad():
        # Iterate over the test dataloader
        for data in tqdm(test_dataloader, leave=False, total=len(test_dataloader)):
            probe, gallery, is_match, probe_pth, gallery_pths, probe_target, gallery_targets = data

            scores = []
            gallery = torch.cat(gallery, axis=0)
            probe_batch = probe.repeat(gallery.shape[0], 1, 1, 1)

            output1, output_class1, output2, output_class2 = model(probe_batch.to(device), gallery.to(device))

            for i in range(len(probe_batch)):
                a = output1[i].detach().cpu().numpy()
                b = output2[i].detach().cpu().numpy()
                cos_sim = dot(a, b)/(norm(a)*norm(b))
                scores.append([cos_sim, gallery_pths[i][0]])
            
            samples += 1

            topk = sorted(scores, key=lambda element: (element[0]), reverse=True)
            rank_matrix.append([probe_pth[0], topk])

            # -- CALCOLO mAP --
            pos_ranks = []
            probe_id = extract_id_from_imagename(probe_pth[0])
            
            for i, score in enumerate(topk):
                gallery_id = extract_id_from_imagename(score[1])
                if probe_id == gallery_id:
                    pos_ranks.append(i)
                    
            if pos_ranks:
                AP = compute_ap(pos_ranks)  
                AP_scores.append(AP)  

        # -- CALCOLO MATRICE DEI RANK --
        for idx, probe in enumerate(rank_matrix):
            probe_id = extract_id_from_imagename(probe[0])
            
            for i in range(K):
                gallery_id = extract_id_from_imagename(probe[1][i][1])
                
                if probe_id == gallery_id:
                    if i == 0:
                        correct1 += 1; correct2 += 1; correct3 += 1; correct4 += 1; correct5 += 1
                        correct6 += 1; correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 1:
                        correct2 += 1; correct3 += 1; correct4 += 1; correct5 += 1; correct6 += 1
                        correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 2:
                        correct3 += 1; correct4 += 1; correct5 += 1; correct6 += 1; correct7 += 1
                        correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 3:
                        correct4 += 1; correct5 += 1; correct6 += 1; correct7 += 1; correct8 += 1
                        correct9 += 1; correct10 += 1
                    elif i == 4:
                        correct5 += 1; correct6 += 1; correct7 += 1; correct8 += 1; correct9 += 1
                        correct10 += 1
                    elif i == 5:
                        correct6 += 1; correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 6:
                        correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 7:
                        correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 8:
                        correct9 += 1; correct10 += 1
                    elif i == 9:
                        correct10 += 1
                    break

    mAP = np.mean(AP_scores)
    print(f'Total samples: {samples}, Rank #1: {correct1/samples*100.0:.3f}%, Rank #2: {correct2/samples*100.0:.3f}%, Rank #3: {correct3/samples*100.0:.3f}%, Rank #4: {correct4/samples*100.0:.3f}%, Rank #5: {correct5/samples*100.0:.3f}%, Rank #6: {correct6/samples*100.0:.3f}%, Rank #7: {correct7/samples*100.0:.3f}%, Rank #8: {correct8/samples*100.0:.3f}%, Rank #9: {correct9/samples*100.0:.3f}%, Rank #10: {correct10/samples*100.0:.3f}%, mAP:{mAP*100.0:.3f}%')
    return correct1/samples, correct5/samples, correct10/samples, mAP

In [7]:
def training_model(model, train_dataloader, test_dataloader, optimizer, triplet_criterion, n_epochs):
    bestRank1 = 0.0

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0.0

        for batch in tqdm(train_dataloader, leave=False, total=len(train_dataloader)):
            imgs = batch[0].to(device)
            targets = batch[1].to(device)
            
            features, _ = model.forward_one(imgs)
            loss = triplet_criterion(features, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        rank1, rank5, rank10, _ = testing_step(model, test_dataloader)

        models_folder = os.path.join('./output/models/')
        os.makedirs(models_folder, exist_ok=True)

        if rank1 > bestRank1:
            torch.save(model.state_dict(), f'{models_folder}/best_{backbone}_state_r1.bin')
            bestRank1 = rank1

        torch.save(model.state_dict(), f'{models_folder}/best_{backbone}_epoch_{epoch}.bin')

        running_loss = train_loss / len(train_dataloader)
        print(f"Epoch: [{epoch+1}/{n_epochs}]| Loss: {running_loss:.5f}")

## Model Parameters

In [8]:
model = BSN(backbone, img_channels, depth, img_w, img_h, n_classes).to(device)
triplet_criterion = HardTripletLoss(margin=0.3)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Network Training

In [9]:
# Start training model
training_model(model, train_dataloader, test_dataloader, optimizer, triplet_criterion, n_epochs)

Total samples: 410, Rank #1: 42.195%, Rank #2: 53.415%, Rank #3: 60.976%, Rank #4: 66.829%, Rank #5: 69.268%, Rank #6: 72.195%, Rank #7: 74.390%, Rank #8: 76.341%, Rank #9: 77.073%, Rank #10: 78.537%, mAP:26.681%
Epoch: [1/20]| Loss: 0.47719


Total samples: 410, Rank #1: 41.707%, Rank #2: 54.146%, Rank #3: 60.488%, Rank #4: 64.878%, Rank #5: 70.000%, Rank #6: 72.195%, Rank #7: 73.415%, Rank #8: 74.878%, Rank #9: 76.341%, Rank #10: 78.049%, mAP:26.229%
Epoch: [2/20]| Loss: 0.49647


Total samples: 410, Rank #1: 41.951%, Rank #2: 54.878%, Rank #3: 61.220%, Rank #4: 66.829%, Rank #5: 69.512%, Rank #6: 72.439%, Rank #7: 74.390%, Rank #8: 75.610%, Rank #9: 76.585%, Rank #10: 77.805%, mAP:26.772%
Epoch: [3/20]| Loss: 0.47614


Total samples: 410, Rank #1: 43.171%, Rank #2: 55.366%, Rank #3: 62.927%, Rank #4: 66.098%, Rank #5: 69.024%, Rank #6: 70.976%, Rank #7: 73.415%, Rank #8: 74.390%, Rank #9: 76.341%, Rank #10: 78.049%, mAP:27.032%
Epoch: [4/20]| Loss: 0.45340


Total samples: 410, Rank #1: 43.659%, Rank #2: 56.341%, Rank #3: 63.171%, Rank #4: 67.561%, Rank #5: 70.732%, Rank #6: 72.195%, Rank #7: 73.659%, Rank #8: 74.634%, Rank #9: 77.073%, Rank #10: 78.537%, mAP:26.809%
Epoch: [5/20]| Loss: 0.44238


...

Total samples: 410, Rank #1: 47.561%, Rank #2: 59.024%, Rank #3: 64.878%, Rank #4: 69.268%, Rank #5: 72.195%, Rank #6: 74.878%, Rank #7: 75.854%, Rank #8: 77.317%, Rank #9: 78.780%, Rank #10: 80.488%, mAP:29.570%
Epoch: [16/20]| Loss: 0.38475


Total samples: 410, Rank #1: 47.805%, Rank #2: 60.488%, Rank #3: 65.854%, Rank #4: 69.268%, Rank #5: 72.439%, Rank #6: 74.878%, Rank #7: 76.341%, Rank #8: 77.561%, Rank #9: 79.268%, Rank #10: 80.000%, mAP:29.902%
Epoch: [17/20]| Loss: 0.38010


Total samples: 410, Rank #1: 47.561%, Rank #2: 59.268%, Rank #3: 65.854%, Rank #4: 69.024%, Rank #5: 72.927%, Rank #6: 75.122%, Rank #7: 75.854%, Rank #8: 77.561%, Rank #9: 79.024%, Rank #10: 80.000%, mAP:30.064%
Epoch: [18/20]| Loss: 0.38402


Total samples: 410, Rank #1: 46.829%, Rank #2: 60.488%, Rank #3: 64.878%, Rank #4: 68.780%, Rank #5: 72.439%, Rank #6: 74.146%, Rank #7: 75.366%, Rank #8: 77.317%, Rank #9: 78.780%, Rank #10: 80.244%, mAP:30.098%
Epoch: [19/20]| Loss: 0.38162


Total samples: 410, Rank #1: 46.829%, Rank #2: 59.756%, Rank #3: 64.146%, Rank #4: 68.537%, Rank #5: 72.683%, Rank #6: 73.902%, Rank #7: 75.610%, Rank #8: 77.805%, Rank #9: 78.293%, Rank #10: 80.244%, mAP:29.831%
Epoch: [20/20]| Loss: 0.37479


 ## Performance Evaluation

In [10]:
# Initialize the Siamese neural network models with the same architecture and parameters
m1 = BSN(backbone, img_channels, depth, img_w, img_h, n_classes).to(device)

# Load pre-trained model weights
m1.load_state_dict(torch.load('./output/models/best_'+backbone+'_state_r1.bin', map_location=device))

# Evaluate and print the performance of the model with the respective evaluation metric
print('Model w/ best R1:')
testing_step(m1, test_dataloader)

Model w/ best R1:


Total samples: 410, Rank #1: 47.805%, Rank #2: 60.488%, Rank #3: 65.854%, Rank #4: 69.268%, Rank #5: 72.439%, Rank #6: 74.878%, Rank #7: 76.341%, Rank #8: 77.561%, Rank #9: 79.268%, Rank #10: 80.000%, mAP:29.902%


(0.47804878048780486, 0.724390243902439, 0.8, np.float64(0.29901631227541436))